In [ ]:
!wget https://blazingsql-colab.s3.amazonaws.com/netflow_data/nf-chunk2.csv

--2020-05-20 05:07:15--  https://blazingsql-colab.s3.amazonaws.com/netflow_data/nf-chunk2.csv
Resolving blazingsql-colab.s3.amazonaws.com (blazingsql-colab.s3.amazonaws.com)... 52.216.168.67
Connecting to blazingsql-colab.s3.amazonaws.com (blazingsql-colab.s3.amazonaws.com)|52.216.168.67|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2725056295 (2.5G) [text/csv]
Saving to: ‘nf-chunk2.csv’

nf-chunk2.csv       100%[===================>]   2.54G  46.9MB/s    in 55s     

2020-05-20 05:08:11 (47.0 MB/s) - ‘nf-chunk2.csv’ saved [2725056295/2725056295]



In [1]:
%%time

from pyspark.sql import SparkSession
from matplotlib import pyplot as plt
import time
import numpy

spark = SparkSession \
 .builder \
 .master("local[*]") \
 .appName("PySpark Netflow Benchmark code") \
 .config("spark.jars.packages","ch.cern.sparkmeasure:spark-measure_2.11:0.13")  \
 .getOrCreate()

CPU times: user 1.3 s, sys: 1.56 s, total: 2.86 s
Wall time: 4.71 s


In [2]:
%%time
# load CSV into Spark
netflow_df = spark.read.format('com.databricks.spark.csv').options(header='true', inferschema='true').load('./data/nf-chunk2.csv')

CPU times: user 13.7 ms, sys: 9.36 ms, total: 23 ms
Wall time: 55.4 s


In [13]:
netflow_cp = netflow_df.union(netflow_df)
#del netflow_df
#netflow_cp1 = netflow_cp.union(netflow_cp)
#del netflow_cp

In [3]:
%%time
# create table for querying
netflow_df.createOrReplaceTempView('netflow')

CPU times: user 356 µs, sys: 331 µs, total: 687 µs
Wall time: 267 ms


In [ ]:
print(netflow_cp1)

DataFrame[TimeSeconds: double, parsedDate: timestamp, dateTimeStr: double, ipLayerProtocol: int, ipLayerProtocolCode: string, firstSeenSrcIp: string, firstSeenDestIp: string, firstSeenSrcPort: int, firstSeenDestPort: int, moreFragments: int, contFragments: int, durationSeconds: int, firstSeenSrcPayloadBytes: int, firstSeenDestPayloadBytes: int, firstSeenSrcTotalBytes: int, firstSeenDestTotalBytes: int, firstSeenSrcPacketCount: int, firstSeenDestPacketCount: int, recordForceOut: int]


In [4]:
result = spark.sql('''SELECT * FROM netflow a''')
print(result);

DataFrame[TimeSeconds: double, parsedDate: timestamp, dateTimeStr: double, ipLayerProtocol: int, ipLayerProtocolCode: string, firstSeenSrcIp: string, firstSeenDestIp: string, firstSeenSrcPort: int, firstSeenDestPort: int, moreFragments: int, contFragments: int, durationSeconds: int, firstSeenSrcPayloadBytes: int, firstSeenDestPayloadBytes: int, firstSeenSrcTotalBytes: int, firstSeenDestTotalBytes: int, firstSeenSrcPacketCount: int, firstSeenDestPacketCount: int, recordForceOut: int]


In [ ]:
%%time

num_runs = 10
queryId = 'TEST_01'
print(queryId)

# make a query
queries = []
queries.append('''SELECT a.firstSeenSrcIP as source FROM netflow a''')
queries.append('''SELECT count(a.firstSeenDestPort) as targetPorts FROM netflow a''')
queries.append('''SELECT SUM(a.firstSeenSrcTotalBytes) as bytesOut FROM netflow a''')
queries.append('''SELECT AVG(a.firstSeenDestTotalBytes) as bytesIn FROM netflow a''')
queries.append('''SELECT DISTINCT(a.durationSeconds) as durationSeconds FROM netflow a''')
queries.append('''SELECT MIN(parsedDate) as firstFlowDate FROM netflow a''')
queries.append('''SELECT MAX(parsedDate) as lastFlowDate FROM netflow a''')
queries.append('''SELECT COUNT(*) as attemptCount FROM netflow a''')

all_times=[]
for query in queries:
    times=[]
    result0 = ""
    print("\nquery:", query)
    for i in range(0,num_runs):
       t0 = time.time()
       result = spark.sql(query)
       t2 = time.time()
       times.append(t2 - t0)
       result0 = result
       del result
       time.sleep(1)
    print(result0)
    all_times.append(times)
    for j in range(0,num_runs):
       print("times[%d]:" %j + str(times[j]))

TEST_01

query: SELECT a.firstSeenSrcIP as source FROM netflow a
DataFrame[source: string]
times[0]:0.18257808685302734
times[1]:0.008485794067382812
times[2]:0.013235807418823242
times[3]:0.00836801528930664
times[4]:0.008410930633544922
times[5]:0.008655548095703125
times[6]:0.008283138275146484
times[7]:0.008509635925292969
times[8]:0.007044315338134766
times[9]:0.0077571868896484375

query: SELECT count(a.firstSeenDestPort) as targetPorts FROM netflow a
DataFrame[targetPorts: bigint]
times[0]:0.04113149642944336
times[1]:0.013326644897460938
times[2]:0.01275944709777832
times[3]:0.01970195770263672
times[4]:0.012926340103149414
times[5]:0.010816097259521484
times[6]:0.01409769058227539
times[7]:0.013797283172607422
times[8]:0.009499311447143555
times[9]:0.021020174026489258

query: SELECT SUM(a.firstSeenSrcTotalBytes) as bytesOut FROM netflow a
DataFrame[bytesOut: bigint]
times[0]:0.020571231842041016
times[1]:0.011971473693847656
times[2]:0.011891365051269531
times[3]:0.0117034912

In [7]:
num_runs = 10
query = '''
        SELECT
            a.firstSeenSrcIp as source,
            a.firstSeenDestIp as destination,
            count(a.firstSeenDestPort) as targetPorts,
            SUM(a.firstSeenSrcTotalBytes) as bytesOut,
            AVG(a.firstSeenDestTotalBytes) as bytesIn,
            SUM(a.durationSeconds) as durationSeconds,
            MIN(parsedDate) as firstFlowDate,
            MAX(parsedDate) as lastFlowDate,
            COUNT(*) as attemptCount
        FROM
            netflow a
        GROUP BY
            a.firstSeenSrcIp,
            a.firstSeenDestIp
            '''

all_times=[]
times=[]
result0 = ""
print("\nquery:", query)
for i in range(0,num_runs):
   t0 = time.time()
   result = spark.sql(query)
   t2 = time.time()
   times.append(t2 - t0)
   result0 = result
   del result
   time.sleep(1)
result0.show()
all_times.append(times)
for j in range(0,num_runs):
   print("times[%d]:" %j + str(times[j]))


query: 
        SELECT
            a.firstSeenSrcIp as source,
            a.firstSeenDestIp as destination,
            count(a.firstSeenDestPort) as targetPorts,
            SUM(a.firstSeenSrcTotalBytes) as bytesOut,
            AVG(a.firstSeenDestTotalBytes) as bytesIn,
            SUM(a.durationSeconds) as durationSeconds,
            MIN(parsedDate) as firstFlowDate,
            MAX(parsedDate) as lastFlowDate,
            COUNT(*) as attemptCount
        FROM
            netflow a
        GROUP BY
            a.firstSeenSrcIp,
            a.firstSeenDestIp
            
+------------+---------------+-----------+--------+-----------------+---------------+-------------------+-------------------+------------+
|      source|    destination|targetPorts|bytesOut|          bytesIn|durationSeconds|      firstFlowDate|       lastFlowDate|attemptCount|
+------------+---------------+-----------+--------+-----------------+---------------+-------------------+-------------------+------------+


In [8]:
for times in (all_times):
    sum = 0.0
    for time_result in (times):
        sum += time_result

    print("총합: " + str(sum))
    print("평균: " + str(numpy.mean(times, keepdims=False)))
    print("분산: " + str(numpy.var(times)))
    print("표준편차: " + str(numpy.std(times)))
    print("\n")

총합: 0.17212939262390137
평균: 0.017212939262390137
분산: 1.733201725642175e-06
표준편차: 0.0013165111946512932


